# 2. HDFS и файловые хранилища

## 2.1 Чтение через файловые форматы

## 1. Поддерживаемые форматы: Parquet, ORC, CSV, JSON

- Колоночные (Parquet/ORC): хранят данные по столбцам, лучше для аналитических запросов
- Строчные (CSV/JSON): хранят данные по строкам, лучше для транзакционных данных

Как работает:

Parquet/ORC хранят статистику (min/max) для каждого столбца в блоке

Spark читает статистику перед чтением данных

Пропускает блоки, не соответствующие условиям WHERE

## 2.  Predicate Pushdown

Parquet и ORC хранят метаданные, такие как минимальное и максимальное значение для каждого столбца в блоке данных. Это означает, что механизм запросов может пропустить чтение блока данных, если видит, что значения в нем не соответствуют запросу. Это называется predicate pushdown.

Запрос: `SELECT l_orderkey FROM lineitem WHERE l_partkey = 17766770`

1. Text gzip data:
   - Время: 11.9 секунд
   - Данных прочитано: 23.7 GB
   - Стоимость: $0.1

2. Parquet gzip (без сортировки):
   - Время: 2.1 секунд (на 82% быстрее)
   - Данных прочитано: 2.0 GB (на 91% дешевле)

3. Parquet gzip (отсортировано по l_partkey):
   - Время: 1.1 секунд (на 90% быстрее)
   - Данных прочитано: 38.8 MB (на 99.9% дешевле)

In [ ]:
# Демонстрация predicate pushdown
# Из статьи: сортировка данных внутри файлов улучшает predicate pushdown

# Пример создания оптимизированной таблицы (адаптировано из статьи)
optimized_df = spark.read.table("customer") \
    .coalesce(4) \
    .sortWithinPartitions("c_name")  # Сортировка внутри партиций улучшает pushdown

# Запись с использованием predicate pushdown оптимизаций
optimized_df.write \
    .format("parquet") \
    .option("compression", "gzip") \
    .saveAsTable("customer_optimized")

print("Таблица создана с оптимизациями для predicate pushdown")

## 3. Column Pruning

Для колоночных форматов уменьшается объем данных, сканируемых из Amazon S3, потому что читаются только данные конкретных столбцов.

In [ ]:
python
# Пример:
# Таблица имеет 100 колонок
# Нам нужны только 3 колонки

# Без column pruning:
df = spark.read.parquet("/data/wide_table.parquet")  # Читает ВСЕ 100 колонок
result = df.select("id", "name", "amount")           # Отбирает 3 после чтения

# С column pruning (автоматически в Parquet/ORC):
result = spark.read.parquet("/data/wide_table.parquet") \
    .select("id", "name", "amount")  # Читает ТОЛЬКО 3 колонки

# Разница в производительности:
# - 100 колонок × 1GB = 100GB данных
# - 3 колонки × 1GB = 3GB данных
# Экономия: 97% данных не читаются!

## 4. Работа с большими директориями

Общая рекомендация - стремиться к splits около 128 MB. Split - это часть файла, например, диапазон байтов в несжатом текстовом файле или страница в Parquet файле.

Возможно такое:

Чтение 100,000 файлов: 11.5 секунд

Чтение 1 файла: 4.3 секунд (на 62% быстрее)

In [ ]:
# Пример проблемы small files:
# /data/sales/
#   ├── part-00000.parquet (1MB)
#   ├── part-00001.parquet (2MB)
#   ├── part-00002.parquet (1.5MB)
#   └── ... 1000 файлов

# Чтение: 1000 обращений к HDFS/S3 вместо 10

# Решение: объединить файлы
from pyspark.sql.functions import spark_partition_id

# Читаем маленькие файлы
small_files_df = spark.read.parquet("/data/sales/")

# Объединяем в оптимальные размеры (128MB - 1GB)
optimal_size_df = small_files_df.repartition(10)  # Разбиваем на 10 партиций

# Сохраняем обратно
optimal_size_df.write \
    .mode("overwrite") \
    .parquet("/data/sales_optimized/")

In [ ]:
# Использование S3DistCP или перезапись данных

def optimize_file_sizes(input_path, output_path):
    """
    Оптимизация размера файлов (адаптация из статьи)
    """
    # Чтение данных с множеством маленьких файлов
    df = spark.read.parquet(input_path)
    
    # Определяем оптимальное количество файлов
    # Из статьи: "aim for splits that are around 128 MB"
    total_size_gb = 74  # Пример из статьи: 74 GB данных
    target_file_size_gb = 0.128  # 128 MB
    optimal_num_files = max(1, int(total_size_gb / target_file_size_gb))
    
    print(f"Оптимальное количество файлов: {optimal_num_files}")
    
    # Объединение файлов (coalesce или repartition)
    optimized_df = df.coalesce(optimal_num_files)
    
    # Запись оптимизированных данных
    optimized_df.write \
        .mode("overwrite") \
        .parquet(output_path)
    
    return optimized_df

# Пример использования
optimized_data = optimize_file_sizes(
    input_path="s3://data/many_small_files/",
    output_path="s3://data/optimized_files/"
)

## 5. Чтение partitioned таблиц

Partitioning делит вашу таблицу на части и хранит связанные данные вместе на основе значений столбцов, таких как дата, страна, регион. Partitions действуют как виртуальные столбцы

Структура partitioned данных:

s3://athena-examples/flight/parquet/

    PRE year=1987/
    
    PRE year=1988/
    
    PRE year=1989/
    ...

In [ ]:
# Создание partitioned таблицы (из статьи)
df.write \
    .partitionBy("year") \
    .parquet("s3://data/flights_partitioned/")

# Чтение с partition pruning


# Эффективный запрос (с partition pruning)
efficient_query = spark.read.parquet("s3://data/flights_partitioned/") \
    .filter("year = 1991")  # Читаются ТОЛЬКО данные за 1991 год

# Неэффективный запрос (без partition pruning)
inefficient_query = spark.read.parquet("s3://data/flights_partitioned/") \
    .filter("dest = 'JFK'")  # Читаются ВСЕ данные, затем фильтруются

print("Partition pruning позволяет читать только нужные partitions")

# Пример производительности из статьи:
# Non-Partitioned Table: 4.8 сек, 74.1 GB данных
# Partitioned Table: 0.7 сек, 29.96 MB данных (99% дешевле, 85% быстрее)

# 2.2 Запись через файловые форматы

## 1. Четыре режима записи:
- `append`: "Добавить данные в существующую таблицу"


- `overwrite`: "Перезаписать таблицу полностью"

- `ignore`: "Пропустить, если таблица существует"

- `error`: "Ошибка, если таблица существует (по умолчанию)

In [ ]:
# Примеры:
df.write.mode("append").parquet("/data/existing_table/")
df.write.mode("overwrite").parquet("/data/table/")
df.write.mode("ignore").parquet("/data/table/")  # Ничего не произойдет

In [ ]:
# Основные параметры записи
optimized_df = spark.read.table("customer") \
    .coalesce(4) \  # Контроль количества файлов
    .sortWithinPartitions("c_name") \  # Сортировка для predicate pushdown
    .write \
    .partitionBy("c_mktsegment", "c_nationkey") \  # Partitioning
    .bucketBy(32, "c_custkey") \  # Bucketing
    .saveAsTable("customer_optimized", format="parquet", compression="gzip")

## 2. Контроль количества файлов: проблема small files

Одна из причин, по которой у вас может оказаться много маленьких файлов - это over-partitioning.

In [ ]:
# Проблема: после groupBy получаем много маленьких файлов
grouped = df.groupBy("category").agg({"amount": "sum"})
print(f"Partitions после groupBy: {grouped.rdd.getNumPartitions()}")

# Решение 1: coalesce (без shuffle)
coalesced = grouped.coalesce(1)  # Объединить в 1 файл
coalesced.write.parquet("/data/coalesced/")

# Решение 2: repartition (с shuffle)
repartitioned = grouped.repartition(5)  # Разбить на 5 файлов
repartitioned.write.parquet("/data/repartitioned/")

# Решение 3: repartition по колонке
repartitioned_by_col = grouped.repartition("category")
repartitioned_by_col.write.parquet("/data/repartitioned_by_col/")

In [ ]:
# Диагностика проблемы small files (основано на статье)
def analyze_small_files_problem(df_path):
    """
    Анализ проблемы small files на основе рекомендаций статьи
    """
    df = spark.read.parquet(df_path)
    
    # Количество партиций
    num_partitions = df.rdd.getNumPartitions()
    
    
    if num_partitions > 1000:
        print("⚠️  ПРОБЛЕМА: Слишком много партиций (>1000)")
        print("   Решение: Уменьшить количество partitions или объединить файлы")
    elif num_partitions > 100:
        print("⚠️  ВНИМАНИЕ: Много партиций (>100)")
        print("   Рекомендация: Рассмотреть объединение файлов")
    else:
        print("✅ Количество партиций в норме")
    
    return num_partitions

# Пример из статьи: сравнение производительности
print("\nПример из статьи:")
print("Запрос: SELECT COUNT(*) FROM lineitem")
print("- 100,000 файлов: 11.5 секунд")
print("- 1 файл: 4.3 секунд (на 62% быстрее)")

In [2]:
# Вставить нашу функцию

## 3. Atomic Commit и надежность записи

In [ ]:
# Проблема: запись может прерваться, оставив частичные данные

# Решение 1: Write Ahead Log (в Spark 3.0+)
spark.conf.set("spark.sql.parquet.writeLegacyFormat", "false")
spark.conf.set("spark.sql.hive.convertMetastoreParquet", "false")

# Решение 2: Staging + Rename
import tempfile
import shutil

def atomic_write(df, output_path):
    """Атомарная запись данных"""
    # 1. Пишем во временную директорию
    temp_dir = tempfile.mkdtemp()
    temp_path = f"{temp_dir}/data"
    
    try:
        df.write.parquet(temp_path)
        
        # 2. Атомарное перемещение
        # В HDFS: rename атомарен
        # В S3: нужен специальный committer
        
        # 3. Удаляем старые данные только после успешной записи
        if os.path.exists(output_path):
            shutil.rmtree(output_path)
        
        shutil.move(temp_path, output_path)
        print(f"✅ Данные записаны атомарно в {output_path}")
        
    except Exception as e:
        print(f"❌ Ошибка записи: {e}")
        # Временные данные автоматически удалятся
    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

# Использование:
atomic_write(df, "/data/final_output/")

In [ ]:
# Исходные данные: 100GB CSV
# Требуется: создать оптимизированную Parquet таблицу


# 1. Читаем исходные CSV данные
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/sales/*.csv")

print(f"Исходные данные: {csv_df.count():,} строк, {csv_df.rdd.getNumPartitions()} партиций")

# 2. Добавляем partitioning колонки (если их нет)
df_with_partitions = csv_df \
    .withColumn("sale_year", year(col("sale_date"))) \
    .withColumn("sale_month", month(col("sale_date"))) \
    .withColumn("sale_day", dayofmonth(col("sale_date")))

# 3. Оптимизируем количество файлов
# Правило: 128MB - 1GB на файл
# Предположим: 100GB данных → ~100 файлов

# Рассчитываем оптимальное количество партиций
total_size_gb = 100  # Предполагаемый размер
target_file_size_gb = 0.5  # Целевой размер файла 500MB
optimal_partitions = max(1, int(total_size_gb / target_file_size_gb))

print(f"Оптимальное количество партиций: {optimal_partitions}")

# 4. Репартиционируем перед записью
optimized_df = df_with_partitions.repartition(optimal_partitions, "sale_year", "sale_month")

# 5. Сортируем данные внутри партиций для лучшего predicate pushdown
sorted_df = optimized_df.sortWithinPartitions("product_id", "customer_id")

# 6. Записываем в Parquet с оптимизациями
sorted_df.write \
    .mode("overwrite") \
    .partitionBy("sale_year", "sale_month") \
    .option("compression", "snappy") \
    .option("parquet.block.size", 256 * 1024 * 1024)  # 256MB блоки \
    .parquet("/data/optimized/sales/")

print("✅ Оптимизированная таблица создана")

# 7. Анализируем результат
from pyspark.sql.functions import count

# Считаем количество файлов
file_count = spark.read.parquet("/data/optimized/sales/") \
    .select(spark_partition_id().alias("partition_id")) \
    .groupBy("partition_id") \
    .agg(count("*").alias("files_in_partition"))

print("Распределение файлов по партициям:")
file_count.show()

## 4. Использование compression

**Сжатие ваших данных может значительно ускорить ваши запросы**. Меньшие размеры данных уменьшают объем данных, сканируемых из S3, что приводит к снижению стоимости выполнения запросов.

Поддерживаемые форматы сжатия из статьи:

- gzip (хорошие коэффициенты сжатия, широкий спектр поддержки)

- Snappy (быстрый)

- zstd (Zstandard - новый формат с хорошим балансом между производительностью и коэффициентом сжатия)

gzip
- Splittable: Нет (для текстовых файлов)

- Лучше для: Хорошие коэффициенты сжатия, широкий спектр поддержки

- Примечание: 
Текстовые файлы, сжатые gzip, нельзя разбивать - нужно читать с начала файла

Snappy
- Splittable: Да (для Parquet/ORC)

- Лучше для: Быстрое сжатие/распаковка

- Примечание из статьи: Splittable in Parquet/ORC formats
Форматы Parquet/ORC со сжатием Snappy можно разбивать на части

zstd (Zstandard)
- Splittable: Да (для Parquet/ORC)

- Лучше для: Баланс между производительностью и коэффициентом сжатия

- Примечание: Более новый формат сжатия

Итог по выбору формата сжатия из статьи:

- gzip: Лучшее сжатие, но текстовые файлы не splittable
- Snappy: Быстрее всего, splittable в Parquet/ORC
- zstd: Хороший баланс, новый формат

In [ ]:

# Пример записи с разными форматами сжатия
df.write \
    .option("compression", "gzip") \  # Из статьи: хорошее сжатие
    .parquet("s3://data/compressed_gzip/")

df.write \
    .option("compression", "snappy") \  # Из статьи: быстрый
    .parquet("s3://data/compressed_snappy/")

## 5. Bucketing 

Еще один способ уменьшить объем данных, которые запрос должен прочитать, - это bucket данные внутри каждой партиции. Bucketing - это техника для распределения записей по отдельным файлам на основе значения одного из столбцов.

In [ ]:
# Пример bucketing из статьи
# Создание таблицы с bucketing (адаптировано)

# Из статьи: "The following table shows the difference in a customer table 
# where the c_custkey column is used to create 32 buckets."

# Не bucketed таблица: 2.29 GB, запрос занимает 1.3 сек
# Bucketed таблица: 72.94 MB, запрос занимает 0.82 сек (на 37% быстрее, на 97% дешевле)

# Создание bucketed таблицы
df.write \
    .bucketBy(32, "customer_id") \  # 32 buckets по customer_id
    .sortBy("customer_name") \  # Сортировка внутри bucket
    .saveAsTable("customers_bucketed")



Bucketing помогает когда
- 1. Есть колонка с высокой кардинальностью (много уникальных значений)
- 2. Многие запросы ищут конкретные значения этой колонки
- 3. Хорошие кандидаты: ID пользователей или устройств

# 2.3 Практика

In [ ]:
### Практическое задание: Создание оптимизированной Parquet таблицы

# Исходные данные: 100GB CSV
# Требуется: создать оптимизированную Parquet таблицу

from pyspark.sql.functions import col, year, month, dayofmonth

# 1. Читаем исходные CSV данные
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/sales/*.csv")

print(f"Исходные данные: {csv_df.count():,} строк, {csv_df.rdd.getNumPartitions()} партиций")

# 2. Добавляем partitioning колонки (если их нет)
df_with_partitions = csv_df \
    .withColumn("sale_year", year(col("sale_date"))) \
    .withColumn("sale_month", month(col("sale_date"))) \
    .withColumn("sale_day", dayofmonth(col("sale_date")))

# 3. Оптимизируем количество файлов
# Правило: 128MB - 1GB на файл
# Предположим: 100GB данных → ~100 файлов

# Рассчитываем оптимальное количество партиций
total_size_gb = 100  # Предполагаемый размер
target_file_size_gb = 0.5  # Целевой размер файла 500MB
optimal_partitions = max(1, int(total_size_gb / target_file_size_gb))

print(f"Оптимальное количество партиций: {optimal_partitions}")

# 4. Репартиционируем перед записью
optimized_df = df_with_partitions.repartition(optimal_partitions, "sale_year", "sale_month")

# 5. Сортируем данные внутри партиций для лучшего predicate pushdown
sorted_df = optimized_df.sortWithinPartitions("product_id", "customer_id")

# 6. Записываем в Parquet с оптимизациями
sorted_df.write \
    .mode("overwrite") \
    .partitionBy("sale_year", "sale_month") \
    .option("compression", "snappy") \
    .option("parquet.block.size", 256 * 1024 * 1024) \  # 256MB блоки
    .parquet("/data/optimized/sales/")

print("✅ Оптимизированная таблица создана")

# 7. Анализируем результат
from pyspark.sql.functions import count

# Считаем количество файлов
file_count = spark.read.parquet("/data/optimized/sales/") \
    .select(spark_partition_id().alias("partition_id")) \
    .groupBy("partition_id") \
    .agg(count("*").alias("files_in_partition"))

print("Распределение файлов по партициям:")
file_count.show()

## 5. Процессы чтения, записи и репликации в HDFS

**Шаги процесса чтения:**

Шаг 1: Клиент запрашивает метаданные у NameNode

Клиент отправляет запрос на чтение файла NameNode

NameNode проверяет права доступа и существование файла

Возвращает: список блоков файла и их расположение на DataNodes

Пример информации: файл из 3 блоков, каждый с 3 репликами на разных DataNodes

Шаг 2: NameNode возвращает информацию о блоках
Для каждого блока NameNode сообщает:

ID блока

Список DataNodes, где хранятся реплики (в порядке близости к клиенту)

Смещение (offset) и размер блока

Типичный ответ: "Блок A находится на DataNode1, DataNode2, DataNode3"

Шаг 3: Клиент читает данные напрямую с DataNodes

Клиент выбирает ближайший доступный DataNode из списка

Устанавливает прямое соединение с выбранным DataNode

Важно: после получения информации от NameNode, клиент общается напрямую с DataNodes

NameNode больше не участвует в процессе чтения

Шаг 4: Последовательное чтение блоков

Клиент читает блоки в порядке их расположения в файле

Каждый блок читается с одного DataNode

Если DataNode недоступен, клиент пробует следующую реплику из списка

Шаг 5: Проверка целостности данных

Каждый блок имеет контрольную сумму (checksum)

DataNode проверяет контрольную сумму при чтении

При обнаружении поврежденных данных клиент запрашивает другую реплику

Шаг 6: Объединение данных

Клиент собирает прочитанные блоки в правильном порядке

Возвращает полный файл или запрошенную часть пользователю

**Шаги процесса записи:**

Шаг 1: Инициализация записи

Клиент запрашивает у NameNode создание нового файла

NameNode проверяет права и добавляет файл в namespace

Важно: файл создается сразу, но пустой

Шаг 2: Получение информации для записи

Клиент отправляет первый пакет данных

NameNode выделяет блоки для записи и выбирает DataNodes для репликации

Выбор DataNodes основывается на:

Свободном месте на узлах

Текущей нагрузке узлов

Rack awareness политике

Шаг 3: Создание pipeline для записи

NameNode возвращает список из 3 DataNodes (по умолчанию)

Клиент создает pipeline (цепочку) из этих DataNodes

Структура pipeline: DataNode1 → DataNode2 → DataNode3

Шаг 4: Запись данных через pipeline

Клиент пишет данные только в первый DataNode в pipeline

DataNode1 получает данные, записывает локально и отправляет DataNode2

DataNode2 получает, записывает и отправляет DataNode3

DataNode3 только записывает (последний в цепочке)

Шаг 5: Подтверждение записи (Acknowledgment)

Подтверждение идет в обратном порядке: DataNode3 → DataNode2 → DataNode1 → Клиент

Только после подтверждения от всех DataNodes блок считается записанным

При ошибке на любом этапе операция повторяется с другими DataNodes

Шаг 6: Завершение записи файла

После записи всех блоков клиент сообщает NameNode о завершении

NameNode финализирует запись в метаданных

Атомарность: файл либо записан полностью, либо не записан вообще

Особенности процесса записи:
Буферизация: данные буферизуются перед отправкой в pipeline

Размер пакета: обычно 64KB пакетами

Идемпотентность: повторная запись того же блока безопасна

Временные файлы: пока файл пишется, он виден только создавшему клиенту

**5. Репликация данных в HDFS**
Политика репликации по умолчанию (3 реплики):
Правило выбора расположения реплик:

Первая реплика:

Если клиент находится внутри кластера: на том же DataNode, где работает клиент

Если клиент вне кластера: случайный DataNode (но не перегруженный)

Вторая реплика:

На другом сервере в другой стойке (rack)

Обеспечивает отказоустойчивость при выходе из строя всей стойки

Третья реплика:

На другом сервере в той же стойке, что и вторая реплика

Уменьшает межстойковый трафик при чтении

Rack Awareness (Осведомленность о стойках):
Зачем нужно:

Предотвращение потери данных при отказе всей стойки

Оптимизация сетевого трафика

Балансировка нагрузки между стойками

Как работает:

Администратор указывает, какие серверы в каких стойках

NameNode знает топологию сети

При выборе DataNodes учитывается принадлежность к стойкам

In [ ]:
Пример топологии:
Стойка 1: [DataNode1, DataNode2, DataNode3]
Стойка 2: [DataNode4, DataNode5, DataNode6] 
Стойка 3: [DataNode7, DataNode8, DataNode9]

Файл с 3 репликами будет размещен:
- Реплика 1: DataNode1 (Стойка 1)
- Реплика 2: DataNode4 (Стойка 2) 
- Реплика 3: DataNode5 (Стойка 2, но другой сервер)

Механизмы репликации:
1. Initial Replication (Первичная репликация)

Происходит при записи файла через pipeline

Все 3 реплики создаются сразу

2. Re-replication (Перерепликация)

Когда DataNode выходит из строя

NameNode обнаруживает недостающие реплики

Запускает процесс создания новых реплик на других DataNodes

Приоритет: сначала реплицируются блоки с наименьшим количеством реплик

3. Decommissioning (Вывод из эксплуатации)

Плановое удаление DataNode из кластера

Блоки с этого узла реплицируются на другие узлы перед удалением

4. Balancer (Балансировщик)

Перемещает блоки между DataNodes для равномерного распределения

Запускается периодически или при разнице в использовании >10%

Мониторинг репликации:
Проблемы, которые отслеживаются:

Under-replicated blocks: блоки с недостаточным количеством реплик

Over-replicated blocks: блоки с избыточным количеством реплик

Mis-replicated blocks: блоки, нарушающие rack awareness политику

Corrupt blocks: поврежденные блоки (обнаруживаются checksum)

Команды для проверки:

## 6. Spark и HDFS: Как они работают вместе

In [ ]:
Архитектура взаимодействия:
Spark Driver (координатор)   ↔   HDFS NameNode (метаданные)
         ↓                            ↑
Spark Executors (рабочие узлы)  ↔   HDFS DataNodes (данные)

**Как Spark читает данные из HDFS:**

Шаг 1: Планирование задач (Task Planning)

Spark Driver запрашивает у NameNode информацию о файле

Получает список всех блоков и их расположение на DataNodes

Ключевой момент: Spark знает, на каких узлах находятся данные

Шаг 2: Создание RDD/DataFrame с учетом локальности

Для каждого блока создается отдельная partition в RDD

Локальность данных: Spark старается запустить задачу на том же узле, где находятся данные

Если узел занят, задача может быть запущена на узле в той же стойке

Шаг 3: Распределение задач по Executors

Driver отправляет задачи (tasks) Executors

Data locality levels (уровни локальности):

PROCESS_LOCAL: данные в памяти того же процесса
NODE_LOCAL: данные на том же узле
RACK_LOCAL: данные в той же стойке
ANY: данные на любом узле
Шаг 4: Параллельное чтение данных

Каждый Executor читает свой блок напрямую с DataNode

Важно: чтение происходит параллельно, без участия Driver

Spark использует нативный Hadoop InputFormat для чтения

Шаг 5: Обработка в памяти

Данные загружаются в память Executors

Обрабатываются согласно логике Spark job

Результаты агрегируются или сохраняются

**Как Spark пишет данные в HDFS:**

Шаг 1: Подготовка данных к записи

Каждый Executor обрабатывает свою часть данных

Результаты остаются в памяти до операции записи

Шаг 2: Определение схемы записи

Spark определяет:

Количество выходных файлов (по количеству partitions)

Формат файлов (Parquet, ORC, CSV, etc.)

Сжатие (Snappy, Gzip, etc.)

Шаг 3: Параллельная запись временных файлов

Каждый Executor пишет свою часть во временный файл в HDFS

Имена временных файлов: _temporary/.../part-XXXXX

Запись происходит напрямую с Executors на DataNodes

Шаг 4: Commit протокол (для надежности)

Важно: Spark использует двухфазный commit для атомарности

Фаза 1: все Executors сообщают Driver, что запись завершена

Фаза 2: Driver выполняет commit, переименовывая временные файлы

При сбое: временные файлы удаляются, операция откатывается

Шаг 5: Финальная запись

Driver переименовывает временные файлы в финальные

Обновляет метаданные в NameNode

Атомарность: либо все файлы записаны, либо ни одного